In [ ]:
# Import paths
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import hstack
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import ndcg_score
import joblib

ROOT = Path.cwd().parents[0]
DATA = ROOT / "models"
OUT  = ROOT / "models"
OUT.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(DATA / "train_program.csv")
test  = pd.read_csv(DATA / "test_program.csv")

print(f"Dataset sizes: train={len(train)}, test={len(test)}")

Dataset sizes: train=299637, test=33113


In [2]:
# Feature engineering: one-hot on interests and field_tags
def to_list(s: str):
    return [x.strip().lower() for x in str(s).split(";") if x.strip()]

mlb_int = MultiLabelBinarizer(sparse_output=True)
mlb_tag = MultiLabelBinarizer(sparse_output=True)

X_train = hstack([
    mlb_int.fit_transform(train["interests"].map(to_list)),
    mlb_tag.fit_transform(train["field_tags"].map(to_list))
], format="csr")
y_train = train["label_match"].values

X_test = hstack([
    mlb_int.transform(test["interests"].map(to_list)),
    mlb_tag.transform(test["field_tags"].map(to_list))
], format="csr")
y_test = test["label_match"].values

print("size:", X_train.shape, X_test.shape)

size: (299637, 48) (33113, 48)


In [3]:
# Random Forest parameter
reg = RandomForestRegressor(
    n_estimators=1000,
    max_depth=20,
    min_samples_split=4,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

In [4]:
# Train
print("Training RandomForestRegressor")
Xtr = X_train.toarray()
Xte = X_test.toarray()
reg.fit(Xtr, y_train)
test_pred = reg.predict(Xte)

Training RandomForestRegressor


In [5]:
# Save artifacts
out_df = test.copy()
out_df["pred_label_match"] = np.round(test_pred, 4)
out_df.to_csv(OUT / "rf_program_test.csv", index=False)

In [6]:
# nDCG
val_df = test[["student_id", "label_match"]].copy()
val_df["pred_label_match"] = np.asarray(test_pred, dtype=float)

rows = []
for sid, g in val_df.groupby("student_id", sort=True):
    y_true = g["label_match"].to_numpy().reshape(1, -1)
    y_pred = g["pred_label_match"].to_numpy().reshape(1, -1)
    rows.append({"student_id": sid, "nDCG@3": float(ndcg_score(y_true, y_pred, k=3))})

ndcg_RF = pd.DataFrame(rows).sort_values("student_id").reset_index(drop=True)
mean_ndcg_RF = float(ndcg_RF["nDCG@3"].mean())

display(ndcg_RF)
print("Mean nDCG@3", round(mean_ndcg_RF, 6))


,student_id,nDCG@3
0,2,0.992900
1,10,0.888877
2,12,0.933226
3,17,0.978671
4,19,1.000000
...,...,...
495,4940,0.837011
496,4966,0.874460
497,4968,0.973260
498,4988,0.884119


Mean nDCG@3 0.928794
